# Experiment 18 — ViT patch-footprint ratio table

For every (foundation model, perturbation patch size $P$) pair, the number
of ViT tokens covered by one perturbation patch at the pipeline's operating
input resolution ($1024\times1024$) is tabulated. Each model's native
patch size and native input resolution determine the effective token size
at $1024\times1024$ after positional-embedding interpolation. The table
allows the suppression-ordering dissociation to be read against the
token-coverage ordering, ruling out the patch-footprint confound.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators_work'))

FM_SPEC = {
    'raddino':    {'native_res': 518, 'patch': 14, 'grid_native': 518 // 14},
    'dinov2':     {'native_res': 518, 'patch': 14, 'grid_native': 518 // 14},
    'dinov3':     {'native_res': 224, 'patch': 16, 'grid_native': 224 // 16},
    'biomedclip': {'native_res': 224, 'patch': 16, 'grid_native': 224 // 16},
    'medsiglip':  {'native_res': 448, 'patch': 14, 'grid_native': 448 // 14},
}
PIPE_RES = 1024
PATCH_SIZES = [4, 8, 16, 32, 64]

rows = []
for m, spec in FM_SPEC.items():
    tok_px = PIPE_RES / spec['grid_native']
    for P in PATCH_SIZES:
        n_tok = (max(P, tok_px) ** 2) / (tok_px ** 2)
        rows.append({'model': m, 'native_res': spec['native_res'],
                     'native_patch': spec['patch'],
                     'grid_at_pipeline': round(PIPE_RES / tok_px, 1),
                     'effective_token_px_at_1024': round(tok_px, 1),
                     'artifact_patch_px': P,
                     'tokens_covered': round(n_tok, 2)})
out = pd.DataFrame(rows)
out_dir = ROOT / 'v4_exp18_patch_footprint'
out_dir.mkdir(parents=True, exist_ok=True)
out.to_parquet(out_dir / 'exp18_patch_footprint.parquet', index=False)
print(out.pivot(index='model', columns='artifact_patch_px',
                values='tokens_covered').round(2))


artifact_patch_px   4    8    16    32    64
model                                       
biomedclip         1.0  1.0  1.0  1.00  1.00
dinov2             1.0  1.0  1.0  1.34  5.35
dinov3             1.0  1.0  1.0  1.00  1.00
medsiglip          1.0  1.0  1.0  1.00  4.00
raddino            1.0  1.0  1.0  1.34  5.35
